# Train Stroke Model — Tiny Transformer

Small Transformer encoder over per-point features for online strokes. Reads from ../data/strokes like the BiGRU notebook.

In [ ]:
from pathlib import Path
import sys, numpy as np, torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
SRC=(Path('..')/'src').resolve(); sys.path.insert(0,str(SRC))
from stroke_recognizer.data import build_dataloaders, DEFAULT_LABELS_HEBREW


In [ ]:
DATA_ROOT=str((Path('..')/'data').resolve())
LABELS=DEFAULT_LABELS_HEBREW
N_POINTS=96; BATCH=64; EPOCHS=30; LR=3e-4; WD=1e-4
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT, device


In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(root=DATA_ROOT, label_names=LABELS, n_points=N_POINTS, batch_size=BATCH)
len(train_loader.dataset), len(val_loader.dataset)


In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, d_in=9, d_model=128, nhead=4, nlayers=2, num_classes=27, dropout=0.1):
        super().__init__()
        self.inp = nn.Linear(d_in, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.inp(x)
        x = self.enc(x)
        x = x.mean(dim=1)
        return self.head(x)

model=TinyTransformer(num_classes=len(LABELS)).to(device)
crit=nn.CrossEntropyLoss(label_smoothing=0.05)
opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
def topk(logits,y,k=1):
    return (logits.topk(k,dim=1).indices==y.view(-1,1)).any(dim=1).float().mean().item()


In [ ]:
hist={'train_loss':[],'val_top1':[]} ; best=0.0
for epoch in range(1,EPOCHS+1):
    model.train(); tl=0.0; n=0
    for x,y in tqdm(train_loader, desc=f'train {epoch}/{EPOCHS}', leave=False):
        x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True)
        logits=model(x); loss=crit(logits,y); loss.backward(); opt.step()
        tl+=loss.item()*x.size(0); n+=x.size(0)
    model.eval(); acc=0.0; m=0
    with torch.no_grad():
        for x,y in tqdm(val_loader, desc='val', leave=False):
            x,y=x.to(device),y.to(device); logits=model(x)
            acc+=topk(logits,y,1)*x.size(0); m+=x.size(0)
    acc/=max(1,m); hist['train_loss'].append(tl/max(1,n)); hist['val_top1'].append(acc)
    print(f'Epoch {epoch:02d} train_loss={hist["train_loss"][-1]:.4f} val_top1={acc:.3f}')

print('Best (not tracked) current val_top1:', hist['val_top1'][-1] if hist['val_top1'] else 0)


In [ ]:
plt.figure(figsize=(8,3)); plt.plot(hist['train_loss']); plt.title('Train loss'); plt.show()
